# Multi-Class Sentiment Classifier (Task 2)

**Goal:** Build a text classifier that reads a sentence and predicts its sentiment.

**Sentiment classes used in this notebook (3-class, multi-class):**
- **Positive** — the writer expresses a good feeling, praise, or satisfaction
- **Negative** — the writer expresses a bad feeling, criticism, or dissatisfaction
- **Neutral** — a plain, factual statement with no emotional tone (dates, prices, schedules, etc.)

**Pipeline:** raw text -> clean text -> remove stop words -> lemmatize -> TF-IDF -> Logistic Regression -> evaluate -> predict on your own sentences -> (optional) Gradio demo.

**About the dataset:** No single public dataset was specified for this internship task, so this notebook builds a small, clearly-labeled *practice dataset* from sentence templates (allowed per the task instructions when no dataset is provided). It is intentionally simple so every step of the pipeline is easy to follow. Section 10 below explains how to swap in a larger real-world dataset (e.g. Twitter Airline Sentiment, Sentiment140) later if you want a stronger submission.


## 1. Install required libraries
Run this once per Colab session.

In [ ]:
# NLTK and scikit-learn already ship with Colab, but this makes sure everything needed is present.
!pip install -q nltk scikit-learn pandas matplotlib seaborn joblib gradio


In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


## 2. Imports

In [ ]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import joblib

random.seed(42)
np.random.seed(42)


## 3. Build the practice dataset

We generate sentences from templates: a **subject** (e.g. "movie", "app") is combined with a **positive/negative adjective** for the Positive/Negative classes, and plain factual templates (dates, prices, schedules) are used for the Neutral class.

This is a *synthetic but clearly labeled* dataset, exactly as allowed by the task instructions when no specific dataset is required. Feel free to add your own sentences to the lists to make it more diverse.

In [ ]:
subjects = ["movie", "product", "restaurant", "service", "app", "book", "trip", "course",
            "teacher", "phone", "laptop", "hotel", "concert", "game", "software", "meal",
            "flight", "class", "team", "event"]

positive_adjs = ["amazing", "fantastic", "wonderful", "excellent", "great", "delightful",
                 "impressive", "outstanding", "brilliant", "lovely", "superb", "fabulous"]
negative_adjs = ["terrible", "awful", "horrible", "disappointing", "poor", "frustrating",
                 "dreadful", "unpleasant", "annoying", "mediocre", "lousy", "unacceptable"]

pos_templates = [
    "The {s} was {a}, I really enjoyed it",
    "I absolutely loved the {s}, it was {a}",
    "What a {a} {s}, highly recommended",
    "This {s} exceeded my expectations, truly {a}",
    "I am so happy with the {s}, it felt {a}",
    "Such a {a} {s}, I would do it again",
]
neg_templates = [
    "The {s} was {a}, I really regret it",
    "I absolutely hated the {s}, it was {a}",
    "What a {a} {s}, would not recommend",
    "This {s} fell short of my expectations, truly {a}",
    "I am so upset with the {s}, it felt {a}",
    "Such a {a} {s}, I would never do it again",
]
neu_templates = [
    "The {s} is scheduled to start at {t}",
    "The {s} was delivered on {d}",
    "This {s} costs around {n} dollars",
    "The {s} lasted about {n} minutes",
    "The report on the {s} was submitted on {d}",
    "The {s} is located on the {n}th floor",
]
times = ["9 AM", "noon", "3 PM", "6 PM", "midnight"]
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

def generate_sentences(templates, adjectives=None, n=90):
    """Fill in templates with random words until we have n unique sentences."""
    sentences = set()
    attempts = 0
    while len(sentences) < n and attempts < 5000:
        attempts += 1
        template = random.choice(templates)
        s = random.choice(subjects)
        if adjectives:
            a = random.choice(adjectives)
            sentence = template.format(s=s, a=a)
        else:
            sentence = template.format(s=s, t=random.choice(times), d=random.choice(days), n=random.randint(2, 120))
        sentences.add(sentence)
    return list(sentences)

positive = generate_sentences(pos_templates, positive_adjs, n=90)
negative = generate_sentences(neg_templates, negative_adjs, n=90)
neutral = generate_sentences(neu_templates, None, n=90)

texts = positive + negative + neutral
labels = (["Positive"] * len(positive)) + (["Negative"] * len(negative)) + (["Neutral"] * len(neutral))

df = pd.DataFrame({"text": texts, "sentiment": labels})
print(f"Total examples: {len(df)}")
df.head()


## 4. Explore the dataset
Always look at your data before modeling — check class balance and read a few samples.

In [ ]:
print(df['sentiment'].value_counts())

plt.figure(figsize=(5,4))
sns.countplot(data=df, x='sentiment', order=['Positive', 'Negative', 'Neutral'])
plt.title('Number of examples per sentiment class')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

df.sample(5, random_state=1)


## 5. Clean and preprocess the text

Steps applied to every sentence:
1. Lowercase everything (so "Great" and "great" are treated the same)
2. Remove punctuation and numbers (keep only letters)
3. Remove **stop words** (common words like "the", "is", "a" that carry little sentiment meaning)
4. **Lemmatize** each remaining word (reduce it to its dictionary/root form, e.g. "loved" -> "love", "movies" -> "movie")

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()                          # 1. lowercase
    text = re.sub(r"[^a-z\s]", "", text)          # 2. keep only letters and spaces
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]  # 3 + 4
    return " ".join(tokens)

df['clean_text'] = df['text'].apply(clean_text)

# Compare a few examples before/after cleaning
df[['text', 'clean_text']].sample(5, random_state=1)


## 6. Split into training and testing sets
We hold out 20% of the data to fairly evaluate the model on sentences it has never seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'],
    test_size=0.2, random_state=42, stratify=df['sentiment']
)

print(f"Training examples: {len(X_train)}")
print(f"Testing examples:  {len(X_test)}")


## 7. Convert text to numbers with TF-IDF

Machine learning models need numbers, not raw text. **TF-IDF** (Term Frequency - Inverse Document Frequency) turns each sentence into a vector of numbers where:
- Words that appear often in a sentence get a higher score (Term Frequency)
- Words that appear in almost every sentence (and are therefore not very informative) get a lower score (Inverse Document Frequency)

`ngram_range=(1, 2)` tells it to consider both single words ("great") and two-word phrases ("not great"), which helps capture more meaning.

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Training matrix shape:", X_train_tfidf.shape)


## 8. Train the classification model

We use **Logistic Regression**, a simple and reliable algorithm that works well for text classification and naturally supports more than 2 classes (multi-class).

*(LinearSVC is an equally good alternative — swap the line below with `from sklearn.svm import LinearSVC` and `model = LinearSVC()` if you want to compare.)*

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
print("Model trained successfully!")


## 9. Evaluate the model
We predict on the held-out test set and calculate the standard classification metrics. These numbers are calculated by the code, not assumed.

In [ ]:
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")


In [ ]:
labels_order = sorted(df['sentiment'].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels_order)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_order, yticklabels=labels_order)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Confusion Matrix - Sentiment Classifier')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
print(classification_report(y_test, y_pred))


## 10. Predict the sentiment of your own sentences

Type any sentence and the model will predict Positive, Negative, or Neutral, along with a confidence score for each class.

In [ ]:
def predict_sentiment(sentence):
    cleaned = clean_text(sentence)
    vec = vectorizer.transform([cleaned])
    prediction = model.predict(vec)[0]
    probabilities = model.predict_proba(vec)[0]
    prob_dict = {cls: round(float(p), 3) for cls, p in zip(model.classes_, probabilities)}
    return prediction, prob_dict

# --- Try it with your own examples ---
test_sentences = [
    "I absolutely loved this, it was fantastic",
    "This was a huge waste of time and money",
    "The meeting starts at 3 PM on Thursday",
]

for sentence in test_sentences:
    pred, probs = predict_sentiment(sentence)
    print(f"Sentence:   {sentence}")
    print(f"Prediction: {pred}")
    print(f"Confidence: {probs}")
    print()


## 11. Save the trained model and vectorizer
Saving both lets you reload them later (e.g. in the Gradio app) without retraining.

In [ ]:
joblib.dump(model, 'sentiment_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("Saved: sentiment_model.pkl, tfidf_vectorizer.pkl")

# To reload later:
# model = joblib.load('sentiment_model.pkl')
# vectorizer = joblib.load('tfidf_vectorizer.pkl')


## 12. (Optional) Simple Gradio demo

This creates a small web form, right inside Colab, where you can type a sentence and instantly see the predicted sentiment. Great for a demo screenshot/video for your supervisor.

In [ ]:
import gradio as gr

def gradio_predict(sentence):
    pred, probs = predict_sentiment(sentence)
    return pred, probs

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(lines=2, placeholder="Type a sentence here, e.g. 'This was a great experience'"),
    outputs=[gr.Textbox(label="Predicted sentiment"), gr.Label(label="Confidence per class")],
    title="Multi-Class Sentiment Classifier",
    description="Enter any sentence to see whether it sounds Positive, Negative, or Neutral.",
)

demo.launch(share=True)  # share=True gives a public link you can click in Colab's output


## 13. (Optional extension) Using a real public dataset instead

To strengthen this project further, you can replace Section 3 with a real dataset, for example:

- **Twitter US Airline Sentiment** (Kaggle) — already has Positive/Negative/Neutral labels
- **Sentiment140** (Kaggle) — 1.6M tweets, Positive/Negative

In Colab:
```python
# Example: upload a kaggle.json API key, then:
# !pip install -q kaggle
# !kaggle datasets download -d crowdflower/twitter-airline-sentiment
# !unzip -q twitter-airline-sentiment.zip
# df = pd.read_csv('Tweets.csv')[['text', 'airline_sentiment']]
# df.columns = ['text', 'sentiment']
```
Everything from Section 4 onward (cleaning, TF-IDF, training, evaluation) works unchanged — just point `df` at the new data with `text` and `sentiment` columns.